## Complete batch in control table

This notebook updates `company_db.batch_control.batch_table` to mark a batch as completed.

It performs the following steps:

* Reads the `id` input parameter for the batch being completed
* Builds a small in-memory dataset with the completed batch status and update timestamp
* Creates a temporary view used as the source for the merge
* Merges the batch status into the control table
* Displays the staged update data for validation
* Queries the control table so the final state can be reviewed

This notebook is typically used near the end of an orchestration flow after the batch processing steps finish successfully.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
# Import pandas to create a one-row in-memory dataset for the batch status update.
import pandas as pd

# Create a text widget so the notebook can receive the batch ID at runtime.
# In job runs, this value is typically passed from an upstream task.
batch_id = dbutils.widgets.text('id',"")

# Read the widget value into a Python variable for use in the staged update.
batch_id = dbutils.widgets.get('id')

# Build a small dataframe representing the completed status for the target batch.
# The timestamp captures when the batch was marked as completed.
df = pd.DataFrame({'batch_id': [batch_id], 'batch_status': ['completed'], 'updated_at': [pd.Timestamp.now()]})

# Convert the pandas dataframe to a Spark dataframe so it can be used in Spark SQL.
df = spark.createDataFrame(df)

# Register the dataframe as a temporary view that will act as the merge source.
df.createOrReplaceTempView("temp_table")

In [ ]:
spark.sql(f"""
MERGE INTO {catalog}.batch_control.batch_table AS t
USING temp_table AS s
ON t.batch_id = s.batch_id

WHEN MATCHED THEN
  UPDATE SET
    t.batch_status = s.batch_status,
    t.updated_at = s.updated_at
-- WHEN NOT MATCHED THEN
--   INSERT (batch_id, batch_status, updated_at,created_at)
--   VALUES (s.batch_id, s.batch_status, s.updated_at,current_timestamp())""")